# Phase 16 — DPO Evaluation & Full Comparison
## CodeMentor-LLM
Three-way comparison:
Base Model vs SFT Model vs DPO Model

## Metrics
- ROUGE-1, ROUGE-2, ROUGE-L
- BERTScore (Precision, Recall, F1)
- Qualitative analysis — 10 examples

# libraries

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes==0.45.3 accelerate==1.5.1 datasets==3.3.2 rouge-score==0.1.2 bert-score==0.3.13

#  Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load models and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
sft_adapter_id = "Abdulmoiz123/codementor-llm-sft"
dpo_adapter_id = "Abdulmoiz123/codementor-llm-dpo"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Base model loaded — {base_model.get_memory_footprint() / 1024**3:.2f} GB")

# Load DPO adapter on top
print("Loading DPO adapter...")
dpo_model = PeftModel.from_pretrained(base_model, dpo_adapter_id)
print("DPO model loaded successfully")

#  Load test dataset and define functions

In [ ]:
from datasets import load_dataset

# Load test split
print("Loading test dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-splits")
test_dataset = dataset["test"].select(range(50))
print(f"Test samples: {len(test_dataset)}")

SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def extract_instruction(text):
    if "<|start_header_id|>user<|end_header_id|>" in text:
        instruction = text.split("<|start_header_id|>user<|end_header_id|>")[-1]
        instruction = instruction.split("<|eot_id|>")[0].strip()
        return instruction
    return ""

def extract_reference(text):
    if "<|start_header_id|>assistant<|end_header_id|>" in text:
        response = text.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        return response
    return ""

def generate_response(model, tokenizer, instruction, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

# Extract instructions and references
instructions = [extract_instruction(s["text"]) for s in test_dataset]
references = [extract_reference(s["text"]) for s in test_dataset]

print(f"Instructions: {len(instructions)}")
print(f"References  : {len(references)}")

# Generate predictions from all 3 models

In [ ]:
print("Generating DPO model predictions...")
dpo_predictions = []
for i, instruction in enumerate(instructions):
    response = generate_response(dpo_model, tokenizer, instruction)
    dpo_predictions.append(response)
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/50")
print(f"DPO predictions done: {len(dpo_predictions)}")

print("\nGenerating SFT model predictions...")
dpo_model.load_adapter(sft_adapter_id, adapter_name="sft")
dpo_model.set_adapter("sft")
sft_predictions = []
for i, instruction in enumerate(instructions):
    response = generate_response(dpo_model, tokenizer, instruction)
    sft_predictions.append(response)
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/50")
print(f"SFT predictions done: {len(sft_predictions)}")

print("\nGenerating Base model predictions...")
dpo_model.disable_adapter_layers()
base_predictions = []
for i, instruction in enumerate(instructions):
    response = generate_response(dpo_model, tokenizer, instruction)
    base_predictions.append(response)
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/50")
dpo_model.enable_adapter_layers()
print(f"Base predictions done: {len(base_predictions)}")

# Compute ROUGE scores for all 3 models

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

def compute_rouge(predictions, references):
    r1, r2, rL = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rL.append(scores['rougeL'].fmeasure)
    return {
        "rouge1": sum(r1)/len(r1),
        "rouge2": sum(r2)/len(r2),
        "rougeL": sum(rL)/len(rL)
    }

# Compute scores
base_rouge = compute_rouge(base_predictions, references)
sft_rouge  = compute_rouge(sft_predictions, references)
dpo_rouge  = compute_rouge(dpo_predictions, references)

# Print comparison
print("ROUGE SCORES — THREE WAY COMPARISON")
print(f"{'Metric':<12} {'Base':>8} {'SFT':>8} {'DPO':>8}")
print("-" * 40)
print(f"{'ROUGE-1':<12} {base_rouge['rouge1']:>8.4f} {sft_rouge['rouge1']:>8.4f} {dpo_rouge['rouge1']:>8.4f}")
print(f"{'ROUGE-2':<12} {base_rouge['rouge2']:>8.4f} {sft_rouge['rouge2']:>8.4f} {dpo_rouge['rouge2']:>8.4f}")
print(f"{'ROUGE-L':<12} {base_rouge['rougeL']:>8.4f} {sft_rouge['rougeL']:>8.4f} {dpo_rouge['rougeL']:>8.4f}")

# Compute BERTScore for all 3 models

In [ ]:
from bert_score import score as bert_score

print("Computing BERTScore...")

P_base, R_base, F1_base = bert_score(base_predictions, references, lang="en", verbose=False)
P_sft,  R_sft,  F1_sft  = bert_score(sft_predictions,  references, lang="en", verbose=False)
P_dpo,  R_dpo,  F1_dpo  = bert_score(dpo_predictions,  references, lang="en", verbose=False)

print("\nBERTScore — THREE WAY COMPARISON")
print(f"{'Metric':<12} {'Base':>8} {'SFT':>8} {'DPO':>8}")
print("-" * 40)
print(f"{'Precision':<12} {P_base.mean():>8.4f} {P_sft.mean():>8.4f} {P_dpo.mean():>8.4f}")
print(f"{'Recall':<12} {R_base.mean():>8.4f} {R_sft.mean():>8.4f} {R_dpo.mean():>8.4f}")
print(f"{'F1':<12} {F1_base.mean():>8.4f} {F1_sft.mean():>8.4f} {F1_dpo.mean():>8.4f}")

# Qualitative comparison — 5 examples

In [ ]:
print("QUALITATIVE COMPARISON — 5 EXAMPLES")

for i in range(5):
    print(f"\nExample {i+1}")
    print(f"INSTRUCTION: {instructions[i]}")
    print(f"\nREFERENCE:\n{references[i][:200]}")
    print(f"\nBASE MODEL:\n{base_predictions[i][:200]}")
    print(f"\nSFT MODEL:\n{sft_predictions[i][:200]}")
    print(f"\nDPO MODEL:\n{dpo_predictions[i][:200]}")
    print("=" * 60)

#  Save evaluation results

In [ ]:
import json

results = {
    "models": {
        "base": "meta-llama/Llama-3.2-3B-Instruct",
        "sft": "Abdulmoiz123/codementor-llm-sft",
        "dpo": "Abdulmoiz123/codementor-llm-dpo"
    },
    "test_samples": 50,
    "rouge_scores": {
        "base": base_rouge,
        "sft": sft_rouge,
        "dpo": dpo_rouge
    },
    "bert_scores": {
        "base": {"precision": P_base.mean().item(), "recall": R_base.mean().item(), "f1": F1_base.mean().item()},
        "sft":  {"precision": P_sft.mean().item(),  "recall": R_sft.mean().item(),  "f1": F1_sft.mean().item()},
        "dpo":  {"precision": P_dpo.mean().item(),  "recall": R_dpo.mean().item(),  "f1": F1_dpo.mean().item()}
    },
    "conclusion": "DPO improves ROUGE-2 and ROUGE-L over SFT. Both SFT and DPO significantly better than base model."
}

with open("dpo_evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to dpo_evaluation_results.json")